# Hardware-Requirement Enrichment Layer (Hardware Top-up)
----------------
**MSc Data Science Dissertation**  
**Author:** Vikrant Deshmukh  
**University:** University of Bristol  
**Project:** Video Game Recommendation System
**MSc Data Science Dissertation**



## Objective

This notebook extends the final Hybrid recommendation system with a
post-recommendation hardware requirement enrichment layer.

The system follows the pipeline:

1. Generate relevant game recommendations using the final Hybrid Recommender.
2. Retrieve minimum and recommended PC hardware requirements for the recommended games.
3. Parse and standardise the retrieved hardware requirements.
4. Cache retrieved Steam requirements to avoid repeated API requests.
5. Convert RAM and storage values into numerical GB values where possible.
6. Attach the structured hardware information to the original Hybrid recommendations while preserving the Hybrid ranking.

The hardware-aware layer does not replace the hybrid recommender. Instead, it acts as a post-recommendation compatibility layer that considers whether relevant games are practical for the user's hardware.

## Importing liabraries and paths

In [1]:
# importing core liabraries
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import time
import requests

from bs4 import BeautifulSoup

In [2]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Configure the project and dataset paths.

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/MSC_DISSERTATION")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Output/live_outputs"

data_path = DATA_DIR / "games_clean_v1.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

games = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", games.shape)

Dataset loaded successfully.
Dataset shape: (89618, 50)


In [6]:
# Main dissertation folders

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MSC_DISSERTATION"
)

DATA_DIR = PROJECT_DIR / "Data"

NOTEBOOK_DIR = (
    PROJECT_DIR
    / "Notebooks"
    / "Implementation Notebooks"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "Output"
    / "evaluation_outputs"
)

# Existing final hybrid notebook

HYBRID_NOTEBOOK_PATH = (
    NOTEBOOK_DIR
    / "Hybrid_Recommender.ipynb"
)

# Persistent Steam hardware-requirements cache

CACHE_PATH = (
    DATA_DIR
    / "hardware_requirements_cache.csv"
)

# Future final hardware-aware output

HARDWARE_OUTPUT_PATH = (
    OUTPUT_DIR
    / "hardware_aware_recommendations.csv"
)

print("Project directory:", PROJECT_DIR)
print("Hybrid notebook:", HYBRID_NOTEBOOK_PATH)
print("Hardware cache:", CACHE_PATH)

Project directory: /content/drive/MyDrive/MSC_DISSERTATION
Hybrid notebook: /content/drive/MyDrive/MSC_DISSERTATION/Notebooks/Implementation Notebooks/Hybrid_Recommender.ipynb
Hardware cache: /content/drive/MyDrive/MSC_DISSERTATION/Data/hardware_requirements_cache.csv


## 1. Load the Final Hybrid Recommender

The hardware-aware system uses the existing final hybrid recommender as its recommendation source.

The hybrid model remains unchanged. Its live `hybrid_recommend()` interface is loaded here so that the hardware layer can operate on recommendations for any eligible catalogue game.

In [7]:
%run "/content/drive/MyDrive/MSC_DISSERTATION/Notebooks/Implementation Notebooks/05_Hybrid_Recommender.ipynb"

print("Hybrid recommender loaded.")

Mounted at /content/drive
Dataset loaded successfully.
Dataset shape: (89618, 50)
All candidate files are available.
Metadata shape: (500, 16)
Graph shape: (500, 16)
Semantic shape: (500, 13)
All candidate files passed validation.
Shared query games: 10
Hybrid configuration is valid.
Metadata fusion shape: (500, 6)
Graph fusion shape: (500, 6)
Semantic fusion shape: (500, 6)
Counter-Strike 2 | recommendations: 10 | runtime: 0.030716 seconds
Grand Theft Auto V Legacy | recommendations: 10 | runtime: 0.020987 seconds
Sid Meier’s Civilization® VI | recommendations: 10 | runtime: 0.01886 seconds
The Witcher 3: Wild Hunt | recommendations: 10 | runtime: 0.022168 seconds
Hollow Knight | recommendations: 10 | runtime: 0.021132 seconds
Stardew Valley | recommendations: 10 | runtime: 0.019239 seconds
Factorio | recommendations: 10 | runtime: 0.019324 seconds
Deep Rock Galactic | recommendations: 10 | runtime: 0.024109 seconds
Phasmophobia | recommendations: 10 | runtime: 0.023502 seconds
Hades 

,query_game,hybrid_rank,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,hybrid_rrf_score
0,Counter-Strike 2,1,Counter-Strike: Source,2.0,1.0,1.0,3,0.016288
1,Counter-Strike 2,2,Splitgate,1.0,5.0,NaN,2,0.012711
2,Counter-Strike 2,3,Tom Clancy's Rainbow Six® Siege,7.0,2.0,NaN,2,0.012422
3,Counter-Strike 2,4,Team Fortress 2,3.0,6.0,NaN,2,0.012410
4,Counter-Strike 2,5,Insurgency,6.0,3.0,NaN,2,0.012410
5,Counter-Strike 2,6,Ironsight,11.0,31.0,34.0,3,0.012157
6,Counter-Strike 2,7,VAIL VR,4.0,13.0,NaN,2,0.011729
7,Counter-Strike 2,8,Warfork,17.0,9.0,NaN,2,0.010992
8,Counter-Strike 2,9,Black Squad,5.0,28.0,NaN,2,0.010699
9,Counter-Strike 2,10,Lightphobe,41.0,4.0,NaN,2,0.010210



Model agreement distribution:
model_agreement
2    73
3    27
Name: count, dtype: int64
Hybrid recommendations saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/hybrid_evaluation_recommendations.csv
Hybrid runtime saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/hybrid_evaluation_runtime.csv
Current | {'metadata': 0.4, 'graph': 0.4, 'semantic': 0.2} | Total: 1.0
Equal | {'metadata': 0.3333333333333333, 'graph': 0.3333333333333333, 'semantic': 0.3333333333333333} | Total: 1.0
Metadata-heavy | {'metadata': 0.5, 'graph': 0.3, 'semantic': 0.2} | Total: 1.0
Graph-heavy | {'metadata': 0.3, 'graph': 0.5, 'semantic': 0.2} | Total: 1.0

Running configuration: Current

Running configuration: Equal

Running configuration: Metadata-heavy

Running configuration: Graph-heavy

Sensitivity result shape: (400, 17)

Recommendations per configuration:
weight_configuration
Current           100
Equal             100
Metadata-heavy    100
Graph-heavy   

,hybrid_rank,query_appid,query_game,appid,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,metadata_rrf,graph_rrf,semantic_rrf,hybrid_rrf_score,weight_configuration,metadata_weight,graph_weight,semantic_weight
200,1,730,Counter-Strike 2,240,Counter-Strike: Source,2.0,1.0,1.0,3,0.008065,0.004918,0.003279,0.016261,Metadata-heavy,0.500000,0.300000,0.200000
257,8,413150,Stardew Valley,1350840,Song Of The Prairie,18.0,33.0,19.0,3,0.006410,0.003226,0.002532,0.012168,Metadata-heavy,0.500000,0.300000,0.200000
226,7,289070,Sid Meier’s Civilization® VI,65980,Sid Meier's Civilization®: Beyond Earth™,9.0,2.0,NaN,2,0.007246,0.004839,0.000000,0.012085,Metadata-heavy,0.500000,0.300000,0.200000
213,4,271590,Grand Theft Auto V Legacy,12210,Grand Theft Auto IV: The Complete Edition,2.0,2.0,NaN,2,0.008065,0.004839,0.000000,0.012903,Metadata-heavy,0.500000,0.300000,0.200000
162,3,427520,Factorio,1760340,ReFactory,5.0,6.0,23.0,3,0.005128,0.005051,0.004016,0.014195,Equal,0.333333,0.333333,0.333333
253,4,413150,Stardew Valley,1062520,Dinkum,3.0,7.0,NaN,2,0.007937,0.004478,0.000000,0.012414,Metadata-heavy,0.500000,0.300000,0.200000
1,2,730,Counter-Strike 2,677620,Splitgate,1.0,5.0,NaN,2,0.006557,0.006154,0.000000,0.012711,Current,0.400000,0.400000,0.200000
116,7,271590,Grand Theft Auto V Legacy,447040,Watch_Dogs® 2,4.0,10.0,NaN,2,0.005208,0.004762,0.000000,0.009970,Equal,0.333333,0.333333,0.333333
2,3,730,Counter-Strike 2,359550,Tom Clancy's Rainbow Six® Siege,7.0,2.0,NaN,2,0.005970,0.006452,0.000000,0.012422,Current,0.400000,0.400000,0.200000
366,7,427520,Factorio,3169270,The Factory Must Grow,33.0,23.0,13.0,3,0.003226,0.006024,0.002740,0.011990,Graph-heavy,0.300000,0.500000,0.200000


Configurations: 4

Queries per configuration:
weight_configuration
Current           10
Equal             10
Graph-heavy       10
Metadata-heavy    10
Name: query_appid, dtype: int64

Unique recommendation counts:
[np.int64(10)]

Duplicate pairs: 0
Self-recommendations: 0

RRF sensitivity output passed validation.
RRF sensitivity results saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/rrf_weight_sensitivity_results.csv
Live source notebooks are available.


Live source recommenders loaded: semantic, graph, metadata
generate_metadata_candidates: True
games exists: True
metadata columns present: True
missing metadata columns: set()
metadata_nn: True
title_to_game_position: True
eligible_core_metadata_matrix: True


,hybrid_rank,appid,recommended_game,hybrid_rrf_score,model_agreement
0,1,2653880,Amber,0.013115,2
1,2,2513170,Origami Lovers,0.012512,2
2,3,2735110,An Eternity Gone By,0.012503,2
3,4,916140,The Tale of Bistun,0.012220,2
4,5,2021040,Shards of Nogard,0.012007,2
5,6,645320,SCARF,0.011120,2
6,7,1015890,TASOMACHI: Behind the Twilight,0.011120,2
7,8,2169570,The Lost Legends of Redwall™: The Scout Anthology,0.011069,2
8,9,843180,Clan O'Conall and the Crown of the Stag,0.010797,2
9,10,1977170,Jusant,0.010658,2


Hybrid recommender loaded.


In [8]:
## testing phase

hybrid_test = hybrid_recommend(
    "red dead redemption 2",
    top_n= 5
)

display(hybrid_test)

,hybrid_rank,query_appid,query_game,appid,recommended_game,metadata_rank,graph_rank,semantic_rank,model_agreement,metadata_rrf,graph_rrf,semantic_rrf,hybrid_rrf_score
0,1,1174180,Red Dead Redemption 2,1404210,Red Dead Online,1.0,1.0,1.0,3,0.006557,0.006557,0.003279,0.016393
1,2,1174180,Red Dead Redemption 2,271590,Grand Theft Auto V Legacy,2.0,2.0,NaN,2,0.006452,0.006452,0.000000,0.012903
2,3,1174180,Red Dead Redemption 2,287700,METAL GEAR SOLID V: THE PHANTOM PAIN,4.0,3.0,NaN,2,0.006250,0.006349,0.000000,0.012599
3,4,1174180,Red Dead Redemption 2,360430,Mafia III: Definitive Edition,5.0,4.0,NaN,2,0.006154,0.006250,0.000000,0.012404
4,5,1174180,Red Dead Redemption 2,447040,Watch_Dogs® 2,6.0,12.0,NaN,2,0.006061,0.005556,0.000000,0.011616


## 2. Steam Hardware Requirement Retrieval

The hardware-aware layer retrieves PC system requirements using the Steam Store API and Steam AppIDs produced by the hybrid recommender.

Only games returned by the hybrid recommender are queried. Retrieved requirements are stored in a persistent local cache to avoid unnecessary repeated API requests.

In [9]:

def clean_requirement_html(html_text):
    if not html_text:
        return None

    soup = BeautifulSoup(html_text, "html.parser")
    return soup.get_text(separator="\n", strip=True)

In [10]:
def fetch_requirements(appid):

  url = "https://store.steampowered.com/api/appdetails"

  paremeters = {"appids": int(appid),
                "l": "en"}

  response = requests.get(url, params=paremeters, timeout = 20)

  data = response.json()
  game = data[str(int(appid))]

  if not game["success"]:
    return None

  game_data = game["data"]

  pc_requirements = game_data.get("pc_requirements", {})

  return {
        "appid": int(appid),
        "game_name": game_data.get("name"),
        "minimum_requirements": clean_requirement_html(
            pc_requirements.get("minimum")
        ),
        "recommended_requirements": clean_requirement_html(
            pc_requirements.get("recommended")
        )
    }

In [11]:
fetch_requirements(1245620)

{'appid': 1245620,
 'game_name': 'ELDEN RING',
 'minimum_requirements': 'Minimum:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10\nProcessor:\nINTEL CORE I5-8400 or AMD RYZEN 3 3300X\nMemory:\n12 GB RAM\nGraphics:\nNVIDIA GEFORCE GTX 1060 3 GB or AMD RADEON RX 580 4 GB\nDirectX:\nVersion 12\nStorage:\n60 GB available space\nSound Card:\nWindows Compatible Audio Device\nAdditional Notes:',
 'recommended_requirements': 'Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10/11\nProcessor:\nINTEL CORE I7-8700K or AMD RYZEN 5 3600X\nMemory:\n16 GB RAM\nGraphics:\nNVIDIA GEFORCE GTX 1070 8 GB or AMD RADEON RX VEGA 56 8 GB\nDirectX:\nVersion 12\nStorage:\n60 GB available space\nSound Card:\nWindows Compatible Audio Device\nAdditional Notes:'}

## 3. Hardware Requirements Cache

Previously retrieved Steam hardware requirements are stored in a local cache.

This avoids repeated API requests for games whose requirements have already been fetched. If the cache file does not exist, an empty cache is created automatically.

In [12]:
def load_hardware_cache():

  if os.path.exists(CACHE_PATH):
        cache = pd.read_csv(CACHE_PATH)

  else:
        cache = pd.DataFrame(
            columns=[
                "appid",
                "game_name",
                "minimum_requirements",
                "recommended_requirements"
            ]
        )

  return cache

In [13]:
def is_game_cached(appid, cache):

    return int(appid) in cache["appid"].astype(int).values

In [14]:
hardware_cache = load_hardware_cache()

is_game_cached(
    1245620,
    hardware_cache
)

True

### Add Missing Game to Cache

If a game's AppID is not already cached, its hardware requirements are fetched from Steam and added to the local cache.

In [15]:
def add_game_to_cache(appid):

  cache = load_hardware_cache()

  if is_game_cached(appid, cache):
    print("Game details have already been cached")

    return cache

  new_game = fetch_requirements(appid)
  new_row = pd.DataFrame([new_game])

  cache = pd.concat(
        [cache, new_row],
        ignore_index=True
    )

  cache.to_csv(
        CACHE_PATH,
        index=False
    )

  print("Game added to cache.")

  return cache



In [16]:
hardware_cache = add_game_to_cache(1245620)

Game details have already been cached


In [17]:
def update_hardware_cache(appids):

    cache = load_hardware_cache()

    for appid in appids:

        if not is_game_cached(appid, cache):

            new_game = fetch_requirements(appid)

            new_row = pd.DataFrame([new_game])

            cache = pd.concat(
                [cache, new_row],
                ignore_index=True
            )

            time.sleep(0.4)

    cache.to_csv(
        CACHE_PATH,
        index=False
    )

    return cache

In [18]:
test_appids = hybrid_test["appid"].tolist()

hardware_cache = update_hardware_cache(
    test_appids
)

print("Cached games:", len(hardware_cache))

Cached games: 83


In [19]:
hardware_results = hybrid_test.merge(
    hardware_cache,
    on="appid",
    how="left"
)

In [20]:
hardware_results[
    [
        "hybrid_rank",
        "recommended_game",
        "minimum_requirements",
        "recommended_requirements"
    ]
]

,hybrid_rank,recommended_game,minimum_requirements,recommended_requirements
0,1,Red Dead Online,Minimum:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10 - 64-bit\nProcessor:\nIntel® Core™ i5-25...,Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10 - April 2018 Update (v1803)\nProcess...
1,2,Grand Theft Auto V Legacy,Minimum:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10 64 Bit\nProcessor:\nIntel Core 2 Quad CP...,Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10 64 Bit\nProcessor:\nIntel Core i5 34...
2,3,METAL GEAR SOLID V: THE PHANTOM PAIN,"Minimum:\nOS:\nWindows 7x64, Windows 8x64, Windows 10x64 (64-bit OS Required)\nProcessor:\nIntel Core i5-4460 (3.40 ...","Recommended:\nOS:\nWindows 7x64, Windows 8x64, Windows 10x64 (64-bit OS Required)\nProcessor:\nIntel Core i7-4790 (3..."
3,4,Mafia III: Definitive Edition,"Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64-bit\nProcessor:\nIntel I5-2500K, AMD...","Recommended:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64-bit\nProcessor:\nIntel I7-3770, ..."
4,5,Watch_Dogs® 2,"Minimum:\nOS *:\nOriginally released for Windows 7, the game can be played on Windows 10 and Windows 11 OS\nProcesso...","Recommended:\nOS *:\nOriginally released for Windows 7, the game can be played on Windows 10 and Windows 11 OS\nProc..."


## 4. Parse Hardware Requirements

Steam provides hardware requirements as semi-structured text.  
The following functions extract individual hardware fields such as operating system, processor, memory, graphics card and storage.

In [21]:
def extract_requirement_field(text, field_name):

    if not isinstance(text, str) or not text.strip():
        return None

    pattern = rf"{re.escape(field_name)}:\s*(.+)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return None

In [ ]:
hardware_results

## 5. Dynamic Hardware Requirement Retrieval

This function connects the hybrid recommender with the hardware-requirement pipeline.

For any valid input game, it generates hybrid recommendations, checks the hardware cache, fetches missing Steam requirements, and returns the recommendations together with their hardware information.

In [22]:
def get_hardware_requirements(input_game, top_n):

  ## get hybrid recommendations

  hybrid_results = hybrid_recommend(
      input_game,
      top_n=top_n
  )

  ## Extract AppIDs
  appids = hybrid_results["appid"].tolist()

  ## Update hardware cache
  hardware_cache = update_hardware_cache(appids)

  # 4. Merge hybrid results with requirements
  results = hybrid_results.merge(
        hardware_cache,
        on="appid",
        how="left"
    )

  return results

In [23]:
## sample check

hardware_test = get_hardware_requirements(
    'Need for Speed™ Most Wanted',
    top_n=10
)

In [25]:
hardware_test[
    [
        "hybrid_rank",
        "recommended_game",
        "minimum_requirements",
        "recommended_requirements"
    ]
]

,hybrid_rank,recommended_game,minimum_requirements,recommended_requirements
0,1,Need For Speed: Hot Pursuit,"OS *:\nWindows XP SP3, Windows XP 64-bit SP2, Windows Vista SP2 (32- or 64-bit), or Windows 7 (32- or 64-bit). (Not ...",NaN
1,2,Burnout™ Paradise Remastered,"Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7, 8.1, 10 64-bit\nProcessor:\nIntel i3 2...",Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10\nProcessor:\nIntel i5 3570K or AMD R...
2,3,Need for Speed™,Minimum:\nRequires a 64-bit processor and operating system\nOS *:\n64-bit Windows 7 or later\nProcessor:\nIntel Core...,Recommended:\nRequires a 64-bit processor and operating system\nOS *:\n64-bit Windows 7 or later\nProcessor:\nIntel ...
3,4,Need for Speed™ Rivals,Minimum:\nOS *:\nWindows 7 (Service Pack 2) 32-Bit\nProcessor:\nIntel 2.4 GHz Core 2 Duo or AMD 2.8 GHz Athlon X2\nM...,Recommended:\nOS *:\nWindows 7 (Service Pack 2)\nProcessor:\nIntel Quad-Core CPU or AMD Six Core CPU\nMemory:\n8 GB ...
4,5,Gas Guzzlers: Combat Carnage,Minimum:\nOS *:\nVista(TM)/Windows 7/Windows 8/Windows 10\nProcessor:\nDual-core 2.0 GHz\nMemory:\n3 MB RAM\nGraphic...,Recommended:\nOS *:\nVista(TM)/Windows 7/Windows 8/Windows 10\nProcessor:\nQuad-core Q6600 2.4 GHz or Equivalent\nMe...
5,6,FlatOut 4: Total Insanity,Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows ® 7 64bit\nProcessor:\nIntel® Core i3 / A...,Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows® 10 64bit\nProcessor:\nIntel® Core i5 /...
6,7,GRIP: Combat Racing,Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64bit\nProcessor:\nIntel Core i3-3220 o...,Recommended:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64bit or Newer\nProcessor:\nIntel C...
7,8,Diesel Guns,Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64-bit 7 or Newer\nProcessor:\nQuad-cor...,Recommended:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 64-bit or Newer\nProcessor:\nQuad-c...
8,9,Need for Speed™ Heat,Minimum:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10\nProcessor:\nFX-6350 or Equivalent; Core...,Recommended:\nRequires a 64-bit processor and operating system\nOS:\nWindows 10\nProcessor:\nRyzen 3 1300X or Equiva...
9,10,Outrunner: Neon Nights,Minimum:\nRequires a 64-bit processor and operating system\nOS *:\nWindows 7 or higher\nProcessor:\nIntel Core i5-25...,Recommended:\nRequires a 64-bit processor and operating system


## 6. Parsing Hardware Requirements

The raw Steam requirement text is converted into structured hardware fields.  
CPU and GPU requirements are retained as text, while RAM and storage are converted into numeric GB values where possible.

In [26]:
def extract_gb(text):

  if not text:
    return None

  match = re.search(
      r"(\d+(?:\.\d+)?)\s*(GB|MB)",
      text,
      flags=re.IGNORECASE
  )

  if not match:
    return None

  value = float(match.group(1))
  unit = match.group(2).upper()

  if unit == "MB":
      value = value / 1024

  return value

## 7. Dynamic Hardware Parsing

The retrieved minimum and recommended Steam requirements are dynamically converted into structured hardware fields for every game returned by the hybrid recommender.

In [27]:
def extract_first_available(text, field_names):

    for field in field_names:

        value = extract_requirement_field(
            text,
            field
        )

        if value is not None:
            return value

    return None

In [28]:
def parse_requirements(text):

    memory = extract_first_available(
        text,
        ["Memory", "RAM"]
    )

    storage = extract_first_available(
        text,
        ["Storage", "Hard Disk Space", "Hard Drive"]
    )

    return {
        "os": extract_first_available(
            text,
            ["OS", "OS *"]
        ),

        "cpu": extract_first_available(
            text,
            ["Processor", "CPU"]
        ),

        "ram_gb": extract_gb(memory),

        "gpu": extract_first_available(
            text,
            ["Graphics", "Video Card", "GPU"]
        ),

        "storage_gb": extract_gb(storage)
    }

### Apply Parsing to All Recommended Games

The parser is applied dynamically to every recommended game returned by the hybrid recommender, for both minimum and recommended hardware requirements.

In [29]:
parsed_min = hardware_test["minimum_requirements"].apply(
    parse_requirements
)

parsed_rec = hardware_test["recommended_requirements"].apply(
    parse_requirements
)

In [30]:
parsed_min.head()

,minimum_requirements
0,"{'os': 'Windows XP SP3, Windows XP 64-bit SP2, Windows Vista SP2 (32- or 64-bit), or Windows 7 (32- or 64-bit). (Not..."
1,"{'os': 'Windows 7, 8.1, 10 64-bit', 'cpu': 'Intel i3 2120 @ 3.3GHz or Phenom II X4 965 @ 3.40GHz', 'ram_gb': 4.0, 'g..."
2,"{'os': '64-bit Windows 7 or later', 'cpu': 'Intel Core i3-4130 or equivalent with 4 hardware threads', 'ram_gb': 6.0..."
3,"{'os': 'Windows 7 (Service Pack 2) 32-Bit', 'cpu': 'Intel 2.4 GHz Core 2 Duo or AMD 2.8 GHz Athlon X2', 'ram_gb': 4...."
4,"{'os': 'Vista(TM)/Windows 7/Windows 8/Windows 10', 'cpu': 'Dual-core 2.0 GHz', 'ram_gb': 0.0029296875, 'gpu': 'Direc..."


### Convert Parsed Requirements into Structured Columns

The parsed minimum and recommended requirements are expanded into separate hardware columns for each recommended game.

In [31]:
min_df = pd.DataFrame(parsed_min.tolist()).add_prefix("min_")
rec_df = pd.DataFrame(parsed_rec.tolist()).add_prefix("rec_")

In [ ]:
min_df.head()

### Combine Parsed Requirements with Hybrid Results

The structured minimum and recommended hardware fields are combined with the original hybrid recommendation output.

In [32]:
hardware_parsed = pd.concat(
    [
        hardware_test.reset_index(drop=True),
        min_df.reset_index(drop=True),
        rec_df.reset_index(drop=True)
    ],
    axis=1
)

In [33]:
hardware_parsed[
    [
        "hybrid_rank",
        "recommended_game",
        "min_cpu",
        "min_ram_gb",
        "min_gpu",
        "min_storage_gb",
        "rec_cpu",
        "rec_ram_gb",
        "rec_gpu",
        "rec_storage_gb"
    ]
].head()

,hybrid_rank,recommended_game,min_cpu,min_ram_gb,min_gpu,min_storage_gb,rec_cpu,rec_ram_gb,rec_gpu,rec_storage_gb
0,1,Need For Speed: Hot Pursuit,Intel Core® 2 Duo 2.0 GHZ or AMD Athlon X2 64 2.4GHZ; 1.5 GB Windows® XP / 2 GB Windows Vista® - Windows 7®,1.00000,DirectX® 9.0c Compatible 3D-accelerated 256 MB video card with Shader Model 3.0* or higher,8.0,None,NaN,None,NaN
1,2,Burnout™ Paradise Remastered,Intel i3 2120 @ 3.3GHz or Phenom II X4 965 @ 3.40GHz,4.00000,NVidia GT 450 or ATI Radeon HD 5750,8.0,Intel i5 3570K or AMD Ryzen 3 1300X,8.000000,Nvidia GTX 750 Ti or AMD Radeon R7 265,8.0
2,3,Need for Speed™,Intel Core i3-4130 or equivalent with 4 hardware threads,6.00000,"NVIDIA GeForce GTX 750 Ti 2GB, AMD Radeon HD 7850 2GB, or equivalent DX11 compatible GPU with 2GB of memory",30.0,Intel Core i5-4690 or equivalent with 4 hardware threads,8.000000,"NVIDIA GeForce GTX 970 4GB, AMD Radeon R9 290 4GB, or equivalent DX11 compatible GPU with 4GB of memory",30.0
3,4,Need for Speed™ Rivals,Intel 2.4 GHz Core 2 Duo or AMD 2.8 GHz Athlon X2,4.00000,AMD Radeon 3870 512 MB or higher performance; NVIDIA GeForce 8800 GT or higher performance; Intel HD 4000 Integrated...,30.0,Intel Quad-Core CPU or AMD Six Core CPU,8.000000,AMD Radeon 7870 3GB or higher performance; NVIDIA GeForce GT660 3GB or higher performance,30.0
4,5,Gas Guzzlers: Combat Carnage,Dual-core 2.0 GHz,0.00293,DirectX(R) 9 Compatible Graphics Card with SM 3.0. (GeForce(R) 8800 Ultra or 4850 AMD/ATi) with 512 MB RAM,7.0,Quad-core Q6600 2.4 GHz or Equivalent,0.003906,"DirectX(R) 9 Compatible Graphics Card (Radeon(R) 6850, GeForce(R) 560) with 1 GB RAM",7.0


In [34]:
def get_hardware_topup(input_game, top_n=10):

    # Hybrid recommendations + raw Steam requirements
    results = get_hardware_requirements(
        input_game,
        top_n=top_n
    )

    # Parse minimum requirements
    parsed_min = results["minimum_requirements"].apply(
        parse_requirements
    )

    min_df = pd.DataFrame(
        parsed_min.tolist()
    ).add_prefix("min_")

    # Parse recommended requirements
    parsed_rec = results["recommended_requirements"].apply(
        parse_requirements
    )

    rec_df = pd.DataFrame(
        parsed_rec.tolist()
    ).add_prefix("rec_")

    # Combine everything
    final_results = pd.concat(
        [
            results.reset_index(drop=True),
            min_df,
            rec_df
        ],
        axis=1
    )

    return final_results

In [35]:
final_test = get_hardware_topup(
    "red dead redemption",
    top_n=10
)


In [36]:
final_test[
    [
        "hybrid_rank",
        "recommended_game",
        "min_cpu",
        "min_ram_gb",
        "min_gpu",
        "min_storage_gb",
        "rec_cpu",
        "rec_ram_gb",
        "rec_gpu",
        "rec_storage_gb"
    ]
]

,hybrid_rank,recommended_game,min_cpu,min_ram_gb,min_gpu,min_storage_gb,rec_cpu,rec_ram_gb,rec_gpu,rec_storage_gb
0,1,Mafia II: Definitive Edition,Intel i5-2500K or AMD FX-8120,6.0000,2GB NVIDIA GeForce GTX 660 or 2GB AMD Radeon HD7870,50.000000,Intel i7-3770 or AMD FX 8350,8.0,4GB NVIDIA GeForce GTX 780 or 4GB AMD Radeon R9 290X,50.0
1,2,Grand Theft Auto: Vice City – The Definitive Edition,Intel® Core™ i5-6600K / AMD FX-6300,8.0000,Nvidia GeForce GTX 760 2GB / AMD Radeon R9 280 3GB,10.000000,Processor: Intel® Core™ i7-6600K / AMD Ryzen 5 2600,16.0,Nvidia GeForce GTX 970 4GB / AMD Radeon RX 570 4GB,10.0
2,3,Wild West Old Sam,Intel Dual-Core 2.4 GHz or better,4.0000,NVIDIA GeForce 8800GT or better,1.000000,Intel Dual-Core 2.4 GHz or better,8.0,GTX 670 2GB / AMD R9 280 better,1.0
3,4,EMZOMBED,None,NaN,None,NaN,None,NaN,None,NaN
4,5,Tomb Raider VI: The Angel of Darkness (2003),1.8GHz Processor,0.5000,3D graphics card compatible with DirectX 9.0c,2.000000,None,NaN,None,NaN
5,6,Mafia III: Definitive Edition,"Intel I5-2500K, AMD FX-8120",6.0000,"2GB of Video Memory & NVIDIA GeForce GTX 660, AMD Radeon HD7870",50.000000,"Intel I7-3770, AMD FX 8350 4.0 Ghz",8.0,"4GB of Video Memory & NVIDIA Geforce GTX 780 or GeForce GTX 1060, AMD Radeon R9 290X",50.0
6,7,Mafia II (Classic),Pentium D 3Ghz or AMD Athlon 64 X2 3600+ (Dual core) or higher,1.5000,nVidia GeForce 8600 / ATI HD2600 Pro or better,8.000000,2.4 GHz Quad Core processor,2.0,nVidia GeForce 9800 GTX / ATI Radeon HD 3870 or better,10.0
7,8,MDK,Pentium 300Mhz,0.0625,DirectX,0.634766,None,NaN,None,NaN
8,9,Deadly Tropics,Intel CPU Core i5-2500K 3.3GHz / AMD CPU Phenom II X4 940,4.0000,Nvidia GPU GeForce GTX 660 / AMD GPU Radeon HD 7870,3.000000,Intel CPU Core i7 3770 3.4 GHz / AMD CPU AMD FX-8350 4 GHz,8.0,Nvidia GPU GeForce GTX 770 / AMD GPU Radeon R9 290,3.0
9,10,Doomsday Scavenger | 末日清道夫,Intel Core i5-3570K or AMD FX-8310,4.0000,NVIDIA 9800 GT 1GB / AMD HD 4870 1GB (DX 10、10.1、11),10.000000,Intel Core i7-4790 or AMD Ryzen 3 3200G,8.0,GTX 1060 6GB / GTX 1660 Super or Radeon RX 590,500.0


The hardware layer does not currently compare the retrieved requirements
with a user-supplied hardware profile or perform hardware-based re-ranking.
It acts as a post-recommendation enrichment layer.